# Online Histogram

A histogram of a stream in a fixed number of bins with `KHist` from [OnlineStats.jl](https://github.com/joshday/OnlineStats.jl), which follows the streaming histogram of Ben-Haim and Tom-Tov (2010): each new value becomes a bin, and the two closest bins merge.

In [ ]:
Pollis.packages("OnlineStats", "Plots")

In [ ]:
using OnlineStats, Random, Statistics, Plots

Random.seed!(1)  # simulated: a stream of 10,000 draws from Exponential(1), whose mean and variance are 1 and median log(2) = 0.693
data = randexp(10_000)  # Replace data with your own stream of observations.

## Bins From a Stream

`value` returns the bin centres and their counts. The centres crowd where the data are dense.

In [ ]:
o = fit!(KHist(20), data)
centres, counts = value(o)
println("Centres: ", round.(centres, digits=3))
println("Counts:  ", counts)
println("Observations: ", nobs(o))

## Plotting the Histogram

In [ ]:
plot(o, xlabel="Value", label="KHist(20)")

## Density Against the True Density

`OnlineStats.pdf` interpolates between the bins. The true density of Exponential(1) is exp(-x).

In [ ]:
xs = 0.25:0.25:4
estimated = [OnlineStats.pdf(o, x) for x in xs]
plot(xs, [estimated exp.(-xs)], label=["KHist(20)" "exp(-x)"], xlabel="x", ylabel="Density")

## Quantiles and Moments From the Bins

`mean`, `var` and `quantile` of a `KHist` are approximations computed from the bins.

In [ ]:
println("Mean from bins:     ", mean(o), "   batch: ", mean(data))
println("Variance from bins: ", var(o), "   batch: ", var(data))
println("Median from bins:   ", median(o), "   batch: ", median(data))

## More Bins, Closer Shape

More bins follow the distribution more closely, in particular in the tail.

In [ ]:
for k in (10, 20, 100)
    h = fit!(KHist(k), data)
    println(k, " bins: 0.95 quantile = ", quantile(h, 0.95), "   batch: ", quantile(data, 0.95))
end

## Merging Two Histograms

Histograms built on separate parts of the stream combine with `merge!`, which inserts the bins of one into the other.

In [ ]:
h1 = fit!(KHist(20), data[1:5_000])
h2 = fit!(KHist(20), data[5_001:end])
merge!(h1, h2)
println("Observations in the merged histogram: ", nobs(h1))
println("Median from the merged bins: ", median(h1))